In [2]:
import dolfinx
from Training_utils import train_loader, train
import torch_geometric as tg
import torch
#from FEniCSx_PyTorch_interface import batched_loss_fn, self_supervised_train
#loss_fn = batched_loss_fn()
from SUPG_prediction_models import *


In [3]:
from Training_utils import train_set_wedge as set
from FEniCSx_PyTorch_interface import Data_to_solver, fem_solver, self_supervised_train

from dolfinx.io import XDMFFile
from mpi4py import MPI
from SPDE_problems import int_to_prblm
class batched_loss_fn():
    def __init__(self, set):
        self.fsl = {}
        for G in set:
            num = G.mesh_id[0]
            with XDMFFile(MPI.COMM_WORLD, f"data/training_Set_wedge/mesh_files/mesh_{G.mesh_id[0]}.xdmf", "r") as xdmf:
                mesh = xdmf.read_mesh(name="mesh")


            fs = int_to_prblm(idx=G.prblm_id, mesh=mesh)
            self.fsl[int(G.mesh_id)] = fem_solver(fs)

    def __call__(self, ptr, idx, y):
        loss_vals = [self.fsl[int(idx[i])](y[ptr[i]:ptr[i+1]] ) for i in range(len(ptr)-1)]
        return torch.stack(loss_vals).sum()
    
loss_fn = batched_loss_fn(set)


In [ ]:
batch_size = 5
loader = train_loader(batch_size=batch_size, set='wedge')
model=SigmoidRestriction(GATv2)


optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [ ]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer, factor=0.5, patience=50)


In [42]:

for i in range(1000):
    print(i)
    loss = self_supervised_train(model=model, loader=loader,loss_fn=loss_fn, optimizer=optimizer, device='cpu')
    if loss > curr_loss:
        print(loss)
    elif curr_loss > loss:    
        print(f"new loss: {loss}")
        curr_loss = loss
        torch.save({'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(), 'loss': loss}, "data/models/sigmoid_wedge_GATv2.pth")
        #scheduler.step()

        


0
0.05118204466998577
1
0.05507929716259241
2
0.06047870358452201
3
0.06173652270808816
4
0.07348623499274254
5
0.05853777378797531
6
0.06200266908854246
7
0.059032855555415154
8
0.05323550384491682
9
0.0569562129676342
10
0.05667502526193857
11
0.05347225954756141
12
0.06726592872291803
13
0.05385522451251745
14
0.0592469647526741
15
0.05637566326186061
16
0.07039942126721144
17
0.06464917026460171
18
0.055950534995645285
19
new loss: 0.046248629689216614
20
0.04827486537396908
21
0.054149684961885214
22
0.052389932330697775
23
0.04787184903398156
24
0.04794392688199878
25
0.05065298639237881
26
0.04660415602847934
27
0.047091430984437466
28
0.04682566085830331
29
new loss: 0.044481879100203514
30
0.0571738900616765
31
0.05890906695276499
32
0.06055685272440314
33
0.06264174543321133
34
0.053242082707583904
35
0.05032496643252671
36
0.05272083729505539
37
0.062134366016834974
38
0.05537340836599469
39
0.05186391342431307
40
new loss: 0.0436994144693017
41
0.04664156027138233
42
0.0538

In [ ]:

model = PenaltyRestriction(model=GATv2)


optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
curr_loss = 5


for i in range(2000):
    print(i)
    loss = self_supervised_train(model=model, loader=loader, loss_fn=loss_fn, optimizer=optimizer, device='cpu')
    if loss > curr_loss:
        print(loss)
    elif curr_loss > loss:    
        print(f"new loss: {loss}")
        curr_loss = loss
        torch.save({'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(), 'loss': loss}, "data/models/penalty_wedge_GATv2.pth")
        #scheduler.step()


model = ClampRestriction(model=GATv2)



0
new loss: 3.476175546646118
1
new loss: 3.473598301410675
2
new loss: 3.469807207584381
3
new loss: 3.466344654560089
4
new loss: 3.462831735610962
5
new loss: 3.4583691358566284
6
new loss: 3.4538996815681458
7
new loss: 3.449495315551758
8
new loss: 3.445959746837616
9
new loss: 3.4438198804855347
10
new loss: 3.442073702812195
11
new loss: 3.44048810005188
12
new loss: 3.439095675945282
13
new loss: 3.4378148913383484
14
new loss: 3.4365530014038086
15
new loss: 3.435393512248993
16
new loss: 3.434353828430176
17
new loss: 3.4333195090293884
18
new loss: 3.4322186708450317
19
new loss: 3.431066334247589
20
new loss: 3.429955303668976
21
new loss: 3.4289000630378723
22
new loss: 3.4278236031532288
23
new loss: 3.426747441291809
24
new loss: 3.425625741481781
25
new loss: 3.4244183897972107
26
new loss: 3.423244833946228
27
new loss: 3.4220409989356995
28
new loss: 3.4206568002700806
29
new loss: 3.419092059135437
30
new loss: 3.4172738194465637
31
new loss: 3.4150401949882507
32
ne

In [ ]:
model = ClampRestriction(GATv2)

loader = train_loader(batch_size=5, set='wedge')
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
curr_loss = 2.8



0
1
2
3
4
new loss: 4.994513511657715
5
new loss: 4.911735951900482
6
new loss: 4.841798782348633
7
new loss: 4.782900393009186
8
new loss: 4.7310614585876465
9
new loss: 4.681474924087524
10
new loss: 4.631108641624451
11
new loss: 4.582353949546814
12
new loss: 4.540958523750305
13
new loss: 4.5028159618377686
14
new loss: 4.465663254261017
15
new loss: 4.429664969444275
16
new loss: 4.395054519176483
17
new loss: 4.362005829811096
18
new loss: 4.33099365234375
19
new loss: 4.301599502563477
20
new loss: 4.273137092590332
21
new loss: 4.246107518672943
22
new loss: 4.220063328742981
23
new loss: 4.195030570030212
24
new loss: 4.17117166519165
25
new loss: 4.148848056793213
26
new loss: 4.1284103989601135
27
new loss: 4.109300494194031
28
new loss: 4.09117192029953
29
new loss: 4.073838770389557
30
new loss: 4.057254195213318
31
new loss: 4.041331171989441
32
new loss: 4.026009142398834
33
new loss: 4.011253714561462
34
new loss: 3.997018575668335
35
new loss: 3.9832680225372314
36
ne

In [8]:

for i in range(2000):
    print(i)
    loss = self_supervised_train(model=model, loader=loader, loss_fn=loss_fn, optimizer=optimizer, device='cpu')
    if loss > curr_loss:
        print(loss)
    elif curr_loss > loss:    
        print(f"new loss: {loss}")
        curr_loss = loss
        torch.save({'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(), 'loss': loss}, "data/models/clamp_wedge_GATv2.pth")
        #scheduler.step()

0
3730.0057866573334
1
3.243463546037674
2
3.362354099750519
3
3.4172701835632324
4
3.4384249448776245
5
3.448243021965027
6
3.452605366706848
7
3.4549573063850403
8
3.456353008747101
9
3.457291543483734
10
3.457943320274353
11
3.4583590626716614
12
3.458617925643921
13
3.4587743878364563
14
3.4588736295700073
15
3.458935499191284
16
3.458972752094269
17
3.45899361371994
18
3.4590041041374207
19
3.459007740020752
20
3.459006905555725
21
3.459003269672394
22
3.458997905254364
23
3.4589913487434387
24
3.458984136581421
25
3.4589763283729553
26
3.458968162536621
27
3.458959937095642
28
3.458951950073242
29
3.4589439630508423
30
3.4589362144470215
31
3.4589285850524902
32
3.458920955657959
33
3.458913505077362
34
3.4589062929153442
35
3.458898901939392
36
3.4588916897773743
37
3.458884835243225
38
3.4588780403137207
39
3.4588711857795715
40
3.458864450454712
41
3.458857774734497
42
3.4588509798049927
43
3.4588441848754883
44
3.458837389945984
45
3.4588305354118347
46
3.4588237404823303
47


KeyboardInterrupt: 